# Robustness Ablation: Obfuscation Stress Test

Evaluates all three models on adversarially obfuscated test data, with and without a deobfuscation defense.

**Obfuscation transforms** (applied to the canonical test split):
- Leetspeak substitution (a→4, e→3, etc.) — 30% of transformed words
- Character spacing (hate → h a t e) — 25%
- Character repetition (hate → haatte) — 25%
- Mixed case (hate → hAtE) — 20%
- Each word has 40% probability of being transformed (seeded per-sample for determinism)

**Deobfuscation defense** (rule-based preprocessing):
1. Unicode normalization
2. Collapse spaced characters (h a t e → hate)
3. Lowercase
4. Reverse leetspeak (4→a, 3→e, etc.)
5. Collapse repeated characters (3+ → 1)

No retraining — all models evaluated using their baseline checkpoints and tuned thresholds.

In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path("../../..").resolve()
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PYTHON = sys.executable
SEED = 42

MODEL_NAMES = ['logreg', 'transformer', 'transformer_lora']
MODEL_LABELS = {'logreg': 'TF-IDF + LogReg', 'transformer': 'DistilBERT', 'transformer_lora': 'DistilBERT + LoRA'}
CONDITIONS = ['baseline', 'obfuscated', 'deobfuscated']
CONDITION_LABELS = {'baseline': 'Clean (baseline)', 'obfuscated': 'Obfuscated (no defense)', 'deobfuscated': 'Obfuscated + Deobfuscation'}

for name in MODEL_NAMES:
    model_dir = ROOT / 'data' / 'models' / name
    assert model_dir.exists() and (model_dir / 'data_splits.pkl').exists(), (
        f"Global model not found at {model_dir}. Train it first.")
print('All model checkpoints found.')

## 1. Generate Obfuscated Test Set & Show Examples

In [ ]:
sys.path.insert(0, str(ROOT))
from model.obfuscation import obfuscate_dataset, deobfuscate_dataset
import pickle

# Load test data (same test set across all models)
# NOTE: pickle used here to match existing project convention (data_splits.pkl)
with open(ROOT / 'data' / 'models' / 'logreg' / 'data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
X_test = splits['X_test'].tolist() if hasattr(splits['X_test'], 'tolist') else list(splits['X_test'])
y_test = splits['y_test']
print(f'Test set: {len(X_test)} samples, {y_test.mean():.2%} toxic')

# Generate obfuscated and deobfuscated versions
X_obfuscated = obfuscate_dataset(X_test, seed=SEED)
X_deobfuscated = deobfuscate_dataset(X_obfuscated)

# Show examples
n_changed = sum(1 for a, b in zip(X_test, X_obfuscated) if a != b)
print(f'Obfuscation changed {n_changed}/{len(X_test)} samples ({n_changed/len(X_test):.1%})')
print()
for i in [0, 10, 50, 100, 500]:
    if X_test[i] != X_obfuscated[i]:
        print(f'[{i}] Original:     {X_test[i][:100]}')
        print(f'     Obfuscated:   {X_obfuscated[i][:100]}')
        print(f'     Deobfuscated: {X_deobfuscated[i][:100]}')
        print()

## 2. Run Evaluation (or Load Cached Results)

In [ ]:
results_path = DATA_DIR / 'robustness_results.json'

if not results_path.exists():
    print('Running evaluation...')
    !cd {ROOT} && {PYTHON} -m model.experiments.robustness.run_evaluation
else:
    print(f'Results already exist in {results_path}, skipping.')

with open(results_path) as f:
    results = json.load(f)

print('\nResults loaded for models:', list(results.keys()))

## 3. Quality Metrics Comparison

In [ ]:
metrics_names = ['precision', 'recall', 'f1_score', 'pr_auc']
colors = {'baseline': 'steelblue', 'obfuscated': 'coral', 'deobfuscated': 'seagreen'}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax_idx, model_name in enumerate(MODEL_NAMES):
    ax = axes[ax_idx]
    x = np.arange(len(metrics_names))
    width = 0.25

    for c_idx, condition in enumerate(CONDITIONS):
        vals = [results[model_name][condition][m] for m in metrics_names]
        bars = ax.bar(x + (c_idx - 1) * width, vals, width,
                      label=CONDITION_LABELS[condition], color=colors[condition])
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}',
                    ha='center', fontsize=7)

    ax.set_ylabel('Score')
    ax.set_title(MODEL_LABELS[model_name], fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names, rotation=45)
    ax.set_ylim(0, 1.15)
    ax.legend(loc='lower right', fontsize=7)

plt.suptitle('Robustness: Clean vs Obfuscated vs Deobfuscated', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(ROOT / 'docs' / 'images' / 'robustness_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to docs/images/robustness_comparison.png')

## 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
cmaps = {'baseline': 'Blues', 'obfuscated': 'Oranges', 'deobfuscated': 'Greens'}

for row, model_name in enumerate(MODEL_NAMES):
    for col, condition in enumerate(CONDITIONS):
        ax = axes[row][col]
        cm = np.array(results[model_name][condition]['confusion_matrix'])
        sns.heatmap(cm, annot=True, fmt='d', cmap=cmaps[condition], ax=ax,
                    xticklabels=['Safe', 'Toxic'], yticklabels=['Safe', 'Toxic'])
        t = results[model_name][condition]['threshold']
        ax.set_title(f'{MODEL_LABELS[model_name]}\n{CONDITION_LABELS[condition]} (t={t:.2f})',
                     fontsize=9, fontweight='bold')
        ax.set_ylabel('True' if col == 0 else '')
        ax.set_xlabel('Predicted' if row == 2 else '')

plt.suptitle('Confusion Matrices: Robustness Stress Test', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. F1 Degradation Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(MODEL_NAMES))
width = 0.25

for c_idx, condition in enumerate(CONDITIONS):
    f1s = [results[m][condition]['f1_score'] for m in MODEL_NAMES]
    bars = ax.bar(x + (c_idx - 1) * width, f1s, width,
                  label=CONDITION_LABELS[condition], color=colors[condition])
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}',
                ha='center', fontsize=9)

ax.set_ylabel('F1 Score')
ax.set_title('F1 Score: Clean vs Obfuscated vs Deobfuscated', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_NAMES])
ax.set_ylim(0, 1.1)
ax.legend()
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, label='_nolegend_')

plt.tight_layout()
plt.savefig(str(ROOT / 'docs' / 'images' / 'robustness_f1_degradation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to docs/images/robustness_f1_degradation.png')

## 6. Summary Table

In [ ]:
print('=' * 90)
print('ROBUSTNESS ABLATION SUMMARY (tuned thresholds from clean validation set)')
print('=' * 90)

for model_name in MODEL_NAMES:
    print(f'\n{MODEL_LABELS[model_name]}')
    print('-' * 75)
    print(f'{"Condition":<30} {"Precision":>10} {"Recall":>10} {"F1":>10} {"PR-AUC":>10}')
    print('-' * 75)
    for condition in CONDITIONS:
        m = results[model_name][condition]
        label = CONDITION_LABELS[condition]
        print(f'{label:<30} {m["precision"]:>10.4f} {m["recall"]:>10.4f} '
              f'{m["f1_score"]:>10.4f} {m["pr_auc"]:>10.4f}')
    # Delta rows
    base_f1 = results[model_name]['baseline']['f1_score']
    obf_f1 = results[model_name]['obfuscated']['f1_score']
    deobf_f1 = results[model_name]['deobfuscated']['f1_score']
    print(f'{"Delta (obfuscated)":<30} {"":>10} {"":>10} {obf_f1 - base_f1:>+10.4f}')
    print(f'{"Delta (deobfuscated)":<30} {"":>10} {"":>10} {deobf_f1 - base_f1:>+10.4f}')
    recovery = (deobf_f1 - obf_f1) / (base_f1 - obf_f1) * 100 if base_f1 != obf_f1 else 0
    print(f'{"Recovery rate":<30} {"":>10} {"":>10} {recovery:>9.1f}%')

## 7. Interpretation

### Obfuscation impact (no defense)

| Model | Baseline F1 | Obfuscated F1 | Delta | Relative drop |
|-------|------------|---------------|-------|---------------|
| TF-IDF + LogReg | 0.8054 | 0.7373 | **-6.8pp** | -8.5% |
| DistilBERT | 0.9318 | 0.8395 | **-9.2pp** | -9.9% |
| DistilBERT + LoRA | 0.7929 | 0.7235 | **-6.9pp** | -8.7% |

All three models degrade under obfuscation, with surprisingly similar relative drops (~8-10%). DistilBERT has the largest absolute drop (-9.2pp) because it starts from a much higher baseline and has more to lose. The obfuscation transforms are particularly effective at disrupting recall --- all models miss significantly more toxic content when the text is perturbed.

**TF-IDF + LogReg** loses F1 primarily through recall drop (-9.8pp): character-level perturbations destroy exact n-gram matches, so known toxic phrases become invisible to the vocabulary. Precision drops only slightly (-3.0pp) because safe texts obfuscated still don't match toxic n-grams.

**DistilBERT** loses recall (-6.8pp) and precision (-11.7pp). The precision drop is large because obfuscated text shifts the subword distribution toward unknown tokens, making the model less confident overall and triggering more false positives at the fixed threshold. The model still catches 85% of toxic content even under obfuscation --- substantially better than either alternative.

**DistilBERT + LoRA** shows the same pattern as LogReg: primarily a recall drop (-8.7pp), with moderate precision reduction (-4.1pp). Its lower baseline means it starts from a weaker position.

### Deobfuscation defense effectiveness

| Model | Obfuscated F1 | Deobfuscated F1 | Recovery | Delta from baseline |
|-------|--------------|----------------|----------|-------------------|
| TF-IDF + LogReg | 0.7373 | 0.7555 | **26.8%** | -5.0pp |
| DistilBERT | 0.8395 | 0.9095 | **75.8%** | -2.2pp |
| DistilBERT + LoRA | 0.7235 | 0.7734 | **71.9%** | -2.0pp |

The deobfuscation defense is dramatically more effective for transformers than for TF-IDF:

**Transformers recover ~72-76% of lost performance.** Once the text is cleaned up (leetspeak reversed, spacing collapsed, case normalized), the transformer's semantic understanding reasserts itself. DistilBERT goes from 0.84 to 0.91, nearly matching its clean baseline of 0.93. The remaining 2.2pp gap comes from imperfect deobfuscation (doubled chars not caught by 3+ collapse rule) and information loss from the transforms.

**TF-IDF recovers only 26.8%.** Even after deobfuscation, TF-IDF only goes from 0.74 to 0.76. The problem: deobfuscation introduces its own artifacts (lowercasing all text, collapsing legitimate repeated chars, replacing digits that were originally digits). These artifacts shift the text away from the TF-IDF training distribution. TF-IDF has no semantic fallback --- if the surface form doesn't match a known n-gram, the model is blind.

### Why transformers benefit more from deobfuscation

The key insight is that deobfuscation is an imperfect text normalization. Transformers are robust to minor text variations (they were pre-trained on diverse text), so imperfect normalization still lands in a familiar region of the model's input space. TF-IDF requires exact surface-form matches, so imperfect normalization introduces new mismatches even as it fixes old ones.

This has a practical implication: **deploying a deobfuscation preprocessor in front of a transformer filter is a high-value, low-cost defense** (75.8% recovery, zero retraining). For TF-IDF, the same defense yields diminishing returns (26.8% recovery) because the model's fragility is fundamental, not just a preprocessing gap.